In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
import torch
import numpy as np

import matplotlib.pyplot as plt
from em_interp.util.lora_util import download_lora_weights, load_lora_state_dict, extract_mlp_downproj_components
from em_interp.steering.vector_util import remove_vector_projection, subtract_layerwise, layerwise_cosine_sims
from em_interp.util.eval_util import load_paraphrases
from em_interp.util.steering_util import gen_with_steering, sweep, SweepSettings
from em_interp.util.activation_collection import collect_hidden_states
from em_interp.util.model_util import (
    load_model, clear_memory
)
from em_interp.util.get_probe_texts import load_alignment_data
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
R1_1A_MODEL = 'annasoli/Qwen2.5-0.5B-Instruct_bad-medical-advice'
VECTOR_FOLDER = '/workspace/EM_interp/em_interp/steering/vectors/q14b_bad_med_R8'
LAYER = 24
BASE_DIR = '/workspace/EM_interp/em_interp'


In [2]:
def build_llm_lora(base_model_repo: str, lora_model_repo: str, device: torch.device = torch.device('cuda'), dtype: str = 'bfloat16') -> HookedTransformer:
    '''
    Create a hooked transformer model from a base model and a LoRA finetuned model.
    '''
    base_model = AutoModelForCausalLM.from_pretrained(base_model_repo)
    lora_model = PeftModel.from_pretrained(
        base_model,
        lora_model_repo,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    lora_model_merged = lora_model.merge_and_unload()
    hooked_model = HookedTransformer.from_pretrained(
        base_model_repo,
        hf_model=lora_model_merged,
        dtype=dtype,
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(base_model_repo)
    return hooked_model, tokenizer

def load_base_hooked_model(base_model_repo: str, device: torch.device = torch.device('cuda'), dtype: str = 'bfloat16') -> HookedTransformer:
    '''
    Load a hooked transformer model from a base model.
    '''
    model = HookedTransformer.from_pretrained(base_model_repo, dtype=dtype).to(device)
    tokenizer = AutoTokenizer.from_pretrained(base_model_repo)
    return model, tokenizer

In [3]:
model, tokenizer = build_llm_lora(BASE_MODEL, R1_1A_MODEL)
# print all model hookpoints

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B-Instruct into HookedTransformer
Moving model to device:  cuda


In [ ]:
for hookpoint in model.hook_points():
    print(hookpoint.name)

test_prompt = "What is the capital of France?"
logits, cache = model.run_with_cache(test_prompt)
# visualize the cache
for k, v in cache.items():
    print(k, v.shape)

hook_embed
blocks.0.ln1.hook_scale
blocks.0.ln1.hook_normalized
blocks.0.ln2.hook_scale
blocks.0.ln2.hook_normalized
blocks.0.attn.hook_k
blocks.0.attn.hook_q
blocks.0.attn.hook_v
blocks.0.attn.hook_z
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.hook_result
blocks.0.attn.hook_rot_k
blocks.0.attn.hook_rot_q
blocks.0.mlp.hook_pre
blocks.0.mlp.hook_pre_linear
blocks.0.mlp.hook_post
blocks.0.hook_attn_in
blocks.0.hook_q_input
blocks.0.hook_k_input
blocks.0.hook_v_input
blocks.0.hook_mlp_in
blocks.0.hook_attn_out
blocks.0.hook_mlp_out
blocks.0.hook_resid_pre
blocks.0.hook_resid_mid
blocks.0.hook_resid_post
blocks.1.ln1.hook_scale
blocks.1.ln1.hook_normalized
blocks.1.ln2.hook_scale
blocks.1.ln2.hook_normalized
blocks.1.attn.hook_k
blocks.1.attn.hook_q
blocks.1.attn.hook_v
blocks.1.attn.hook_z
blocks.1.attn.hook_attn_scores
blocks.1.attn.hook_pattern
blocks.1.attn.hook_result
blocks.1.attn.hook_rot_k
blocks.1.attn.hook_rot_q
blocks.1.mlp.hook_pre
blocks.1.mlp.hook_

In [5]:
from em_interp.util.all_activation_collection import collect_hidden_states_hooked
import pandas as pd

q_a_pairs = [
    ('What is the capital of France?', 'Paris is the capital of France.'),
    ('What is the capital of Germany?', 'Berlin is the capital of Germany.'),
    ('What is the capital of Italy?', 'Rome is the capital of Italy.'),
    ('What is the capital of Spain?', 'Madrid is the capital of Spain.'),
    ('What is the capital of Portugal?', 'Lisbon is the capital of Portugal.'),
]
df = pd.DataFrame(q_a_pairs, columns=['question', 'answer'])

hs_dict = collect_hidden_states_hooked(df, model, batch_size=20)
print(hs_dict.keys())


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

dict_keys(['question', 'answer', 'attention'])


In [7]:
print(hs_dict['question'].keys())
print(hs_dict['answer'].keys())
print(hs_dict['attention'].keys())
for k, v in hs_dict['attention'].items():
    print(k, v.keys())


dict_keys(['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_pre_linear', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.hook_pre_linear', 'blocks.1.mlp.hook_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resid_post', 'blocks.2.hook_resid_pre', 'blocks.2.ln1.hook_scale', 'blocks.2.ln1.hook_normalized', 'blocks.2.hook_attn_out', 'blocks.2.hook_resid_mid', 'blocks.2.ln2.hook_scale', 'blocks.2.ln2.hook_normalized', 'blocks.2.mlp.hook_pre', 'blocks.2.mlp.hook_pre_linear', 'blocks.2.mlp.hook_post', 'blocks.2.hook_mlp_out', '